In [ ]:
# YZ50 Demo Day - 4: kendi sesime fine-tune
# Ön eğitilmiş modelden başlar. Veri: 3 Alania dosyası + kendi sesim (6 kez tekrar). Etiket: ME.
# Yarıda kesilirse baştan çalıştır: son checkpoint'ten devam eder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, math, time, glob, json, wave
import numpy as np, torch, torch.nn as nn
import torch.nn.functional as F
from transformers import MimiModel

In [ ]:
# Sözlük: özel token'lar + metin karakterleri + ses token'ları. Bütün notebook'larda aynı.
import unicodedata
PAD, BOS, SEP, EOS, MALE, FEMALE, ME = 0, 1, 2, 3, 4, 5, 6
CHARS = " abcçdefgğhıijklmnoöpqrsştuüvwxyzâîû0123456789.,?!'-:"
CHAR_BASE = 16
stoi = {c: CHAR_BASE + i for i, c in enumerate(CHARS)}
SR, FRAME_RATE = 24000, 12.5             # Mimi codec: 24 kHz ses, saniyede 12.5 kare
N_CB, CB_SIZE = 8, 2048                  # her karede 8 codebook, her biri 2048 kod
AUDIO_BASE = 128
VOCAB = AUDIO_BASE + N_CB * CB_SIZE      # 16512
TOK_PER_S = int(N_CB * FRAME_RATE)       # saniyede 100 ses token'ı
assert CHAR_BASE + len(CHARS) <= AUDIO_BASE

_MAP = str.maketrans({"’": "'", "‘": "'", "“": "", "”": "", '"': "", "–": "-", "—": "-", "…": "."})
def norm_text(s):
    s = unicodedata.normalize("NFC", s).replace("I", "ı").replace("İ", "i").lower().translate(_MAP)
    return " ".join("".join(c for c in s if c in stoi).split())
def encode_text(s):
    return [stoi[c] for c in norm_text(s)]
# Bir örnek: [BOS, ses etiketi, metin karakterleri..., SEP, kare0 cb0..cb7, kare1 cb0..cb7, ..., EOS]

In [ ]:
ROOT = '/content/drive/MyDrive/yz50-demo-day'
TOK_DIR = f'{ROOT}/tokens/alania'
RUN = 'finetune_me'
INIT_FROM = f'{ROOT}/ckpt/pretrain_last.pt'
MAX_FILES, EXTRA = 3, [(f'{ROOT}/tokens/me', 6)]      # 3 alania dosyası + kendi sesim x6
VAL_PATH = f'{ROOT}/tokens/me/val.npz'                # val loss kendi sesimde ölçülür
BLOCK = 2048
N_LAYER, N_HEAD, N_EMBD, DROPOUT = 12, 12, 768, 0.1
BATCH = 16
MAX_STEPS, WARMUP = 1500, 100
MAX_LR, MIN_LR, WEIGHT_DECAY = 1e-4, 1e-5, 0.1
EVAL_EVERY, VAL_BATCHES, CKPT_EVERY = 250, 20, 500
SAMPLE_TEXTS = ['Merhaba, ben sıfırdan eğitilmiş bir konuşma modeliyim.',
                'On iki hafta önce tek bir nöronla başladım.',
                'Hesabınıza bin iki yüz elli lira yatırılmıştır.',
                'Beş bin iki yüz kırk üç numaralı dosyanız onaylandı.']
SAMPLE_TAG, SAMPLE_SECONDS, TEMP, TOP_K = ME, 8, 0.8, 80
DEVICE, COMPILE, AMP = 'cuda', True, True

In [ ]:
# Veri: her örnek bölünmeden, BLOCK+1 uzunluğunda satırlara paketlenir. Boş kalan yer PAD.
def load_tokens(paths):
    toks, lens = [], []
    for p in paths:
        d = np.load(p); toks.append(d['tokens']); lens.append(d['lens'])
    return np.concatenate(toks), np.concatenate(lens)

def pack_rows(tokens, lens, seed):
    starts = np.concatenate([[0], np.cumsum(lens)[:-1]])
    rows, row, fill = [], np.zeros(BLOCK + 1, np.uint16), 0
    for i in np.random.default_rng(seed).permutation(len(lens)):
        n = int(lens[i])
        if n > BLOCK + 1: continue
        if fill + n > BLOCK + 1:
            rows.append(row); row, fill = np.zeros(BLOCK + 1, np.uint16), 0
        row[fill:fill + n] = tokens[starts[i]:starts[i] + n]; fill += n
    rows.append(row)
    return np.stack(rows)

def get_batch(rows, i):
    xb = torch.from_numpy(rows[i:i + BATCH].astype(np.int64)).to(DEVICE)
    x, y = xb[:, :-1], xb[:, 1:].clone()
    y[(y < AUDIO_BASE) & (y != EOS)] = -100          # loss yalnız ses token'ları ve EOS üstünde
    return x, y

train_tok, train_len = load_tokens(sorted(glob.glob(f'{TOK_DIR}/train_*.npz'))[:MAX_FILES])
for d, rep in EXTRA:                                 # ek veri: (dizin, kaç kez tekrar)
    t, l = load_tokens(sorted(glob.glob(f'{d}/train_*.npz')))
    train_tok, train_len = np.concatenate([train_tok] + [t] * rep), np.concatenate([train_len] + [l] * rep)
val_tok, val_len = load_tokens([VAL_PATH])
val_rows = pack_rows(val_tok, val_len, seed=0)
print(f'train: {len(train_len):,} örnek, {len(train_tok):,} token (~{(len(train_tok) - train_len.sum() * 0.12) / TOK_PER_S / 3600:.0f} saat ses) | val: {len(val_len):,} örnek, {len(val_rows)} satır')

In [ ]:
# Model: GPT-2 mimarisi, sıfırdan
class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_attn = nn.Linear(c['n_embd'], 3 * c['n_embd'])
        self.c_proj = nn.Linear(c['n_embd'], c['n_embd']); self.c_proj.RESID = True
        self.n_head, self.dropout = c['n_head'], c['dropout']
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.c_attn(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.dropout if self.training else 0.0)
        return self.c_proj(y.transpose(1, 2).contiguous().view(B, T, C))

class MLP(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c_fc = nn.Linear(c['n_embd'], 4 * c['n_embd'])
        self.c_proj = nn.Linear(4 * c['n_embd'], c['n_embd']); self.c_proj.RESID = True
        self.drop = nn.Dropout(c['dropout'])
    def forward(self, x):
        return self.drop(self.c_proj(F.gelu(self.c_fc(x), approximate='tanh')))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln_1, self.attn = nn.LayerNorm(c['n_embd']), CausalSelfAttention(c)
        self.ln_2, self.mlp = nn.LayerNorm(c['n_embd']), MLP(c)
    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        return x + self.mlp(self.ln_2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c = c
        self.wte = nn.Embedding(c['vocab'], c['n_embd'])
        self.wpe = nn.Embedding(c['block'], c['n_embd'])
        self.h = nn.ModuleList([Block(c) for _ in range(c['n_layer'])])
        self.ln_f = nn.LayerNorm(c['n_embd'])
        self.lm_head = nn.Linear(c['n_embd'], c['vocab'], bias=False)
        self.wte.weight = self.lm_head.weight                      # weight tying
        self.apply(self._init)
    def _init(self, m):
        if isinstance(m, nn.Linear):
            std = 0.02 * (2 * self.c['n_layer']) ** -0.5 if hasattr(m, 'RESID') else 0.02
            nn.init.normal_(m.weight, mean=0.0, std=std)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None, last_only=False):
        B, T = idx.shape
        x = self.wte(idx) + self.wpe(torch.arange(T, device=idx.device))
        for b in self.h: x = b(x)
        x = self.ln_f(x)
        if last_only: return self.lm_head(x[:, -1, :]), None
        logits = self.lm_head(x)
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100)
        return logits, loss

cfg = dict(vocab=VOCAB, block=BLOCK, n_layer=N_LAYER, n_head=N_HEAD, n_embd=N_EMBD, dropout=DROPOUT)

In [ ]:
# Kurulum: model, optimizer, (varsa) checkpoint'ten devam
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True; torch.backends.cudnn.allow_tf32 = True
os.makedirs(f'{ROOT}/ckpt', exist_ok=True); os.makedirs(f'{ROOT}/samples/{RUN}', exist_ok=True)
CKPT, LOG, BEST = f'{ROOT}/ckpt/{RUN}_last.pt', f'{ROOT}/{RUN}_log.json', f'{ROOT}/ckpt/{RUN}_best.pt'
torch.manual_seed(1337)
raw = GPT(cfg).to(DEVICE)
decay = [p for p in raw.parameters() if p.dim() >= 2]
no_decay = [p for p in raw.parameters() if p.dim() < 2]
opt = torch.optim.AdamW([{'params': decay, 'weight_decay': WEIGHT_DECAY}, {'params': no_decay, 'weight_decay': 0.0}],
                        lr=MAX_LR, betas=(0.9, 0.95), eps=1e-8, fused=(DEVICE == 'cuda'))
step, epoch, ptr, log = 0, 0, 0, []
if os.path.exists(CKPT):
    ck = torch.load(CKPT, map_location=DEVICE)
    raw.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
    step, epoch, ptr, log = ck['step'], ck['epoch'], ck['ptr'], ck['log']
    print(f'checkpoint bulundu: adım {step} ile devam')
elif INIT_FROM:
    raw.load_state_dict(torch.load(INIT_FROM, map_location=DEVICE)['model'])
    print('başlangıç ağırlıkları:', INIT_FROM)
model = torch.compile(raw) if COMPILE else raw
print(f'parametre: {sum(p.numel() for p in raw.parameters()):,}')

def get_lr(it):
    if it < WARMUP: return MAX_LR * (it + 1) / WARMUP
    if it >= MAX_STEPS: return MIN_LR
    r = (it - WARMUP) / (MAX_STEPS - WARMUP)
    return MIN_LR + 0.5 * (1 + math.cos(math.pi * r)) * (MAX_LR - MIN_LR)

def autocast():
    return torch.autocast(device_type=DEVICE, dtype=torch.bfloat16, enabled=AMP)

In [ ]:
# Konuşturma: metni ver, ses token'larını tek tek örnekle, Mimi ile dalgaya çevir
mimi = MimiModel.from_pretrained('kyutai/mimi').to(DEVICE).eval()

@torch.no_grad()
def speak(text, tag=SAMPLE_TAG, seconds=SAMPLE_SECONDS, temp=TEMP, top_k=TOP_K, seed=0):
    g = torch.Generator(device=DEVICE).manual_seed(seed)
    x = torch.tensor([[BOS, tag] + encode_text(text) + [SEP]], device=DEVICE)
    out = []
    while len(out) < int(seconds * TOK_PER_S) and x.shape[1] < BLOCK:
        with autocast(): logits, _ = raw(x, last_only=True)
        logits = logits[0].float()
        cb = len(out) % N_CB                                   # sıradaki token hangi codebook'tan
        lo = AUDIO_BASE + cb * CB_SIZE
        mask = torch.full_like(logits, float('-inf')); mask[lo:lo + CB_SIZE] = 0
        if cb == 0 and len(out) >= N_CB: mask[EOS] = 0         # EOS yalnız kare sınırında
        v, ix = torch.topk((logits + mask) / temp, top_k)
        tok = ix[torch.multinomial(F.softmax(v, dim=-1), 1, generator=g)]
        if tok.item() == EOS: break
        out.append(tok.item() - lo)
        x = torch.cat([x, tok.view(1, 1)], dim=1)
    n = len(out) // N_CB
    if n < 2: return np.zeros(SR // 10, dtype=np.float32)
    codes = torch.tensor(out[:n * N_CB], device=DEVICE).view(n, N_CB).T[None]      # (1, 8, kare)
    return mimi.decode(codes).audio_values[0, 0].float().cpu().numpy()

def save_wav(path, a):
    with wave.open(path, 'wb') as w:
        w.setnchannels(1); w.setsampwidth(2); w.setframerate(SR)
        w.writeframes((np.clip(a, -1, 1) * 32767).astype(np.int16).tobytes())

In [ ]:
# Ölçüm: val loss + sabit cümlelerin sesi + log. Checkpoint.
@torch.no_grad()
def evaluate(step, train_loss, tok_s):
    raw.eval()
    nb = max(1, min(VAL_BATCHES, len(val_rows) // BATCH))
    vl = 0.0
    for i in range(nb):
        x, y = get_batch(val_rows, i * BATCH)
        with autocast(): _, loss = model(x, y)
        vl += loss.item() / nb
    files = []
    for k, text in enumerate(SAMPLE_TEXTS):
        a = speak(text)
        name = f'{RUN}_step{step:06d}_{k}.wav'; save_wav(f'{ROOT}/samples/{RUN}/{name}', a); files.append(name)
        try:
            from IPython.display import Audio, display
            display(Audio(a, rate=SR))
        except Exception: pass
    raw.train()
    prev_best = min([l['val'] for l in log], default=float('inf'))
    log.append(dict(step=step, train=train_loss, val=vl, lr=get_lr(step), tok_s=tok_s, samples=files))
    json.dump(dict(run=RUN, cfg=cfg, texts=SAMPLE_TEXTS, log=log), open(LOG, 'w'), ensure_ascii=False)
    if vl < prev_best:                                   # en iyi val loss'lu ağırlıklar ayrıca saklanır
        torch.save(dict(model=raw.state_dict(), step=step, val=vl, cfg=cfg), BEST + '.part'); os.replace(BEST + '.part', BEST)
    print(f'--- adım {step}: val loss {vl:.4f}{"  (en iyi, kaydedildi)" if vl < prev_best else ""} | örnekler: {files}')

def save_ckpt():
    torch.save(dict(model=raw.state_dict(), opt=opt.state_dict(), step=step, epoch=epoch, ptr=ptr, log=log, cfg=cfg), CKPT + '.part')
    os.replace(CKPT + '.part', CKPT)

In [ ]:
# Eğitim
rows = pack_rows(train_tok, train_len, seed=1000 + epoch)
print(f'bir tur = {len(rows) // BATCH:,} adım')
if step == 0: evaluate(0, float('nan'), 0.0)                     # eğitimsiz model: gürültü
raw.train()
t0, s0, recent = time.time(), step, []
while step < MAX_STEPS:
    if ptr + BATCH > len(rows):
        epoch += 1; ptr = 0; rows = pack_rows(train_tok, train_len, seed=1000 + epoch)
    x, y = get_batch(rows, ptr); ptr += BATCH
    lr = get_lr(step)
    for g in opt.param_groups: g['lr'] = lr
    with autocast(): _, loss = model(x, y)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    norm = torch.nn.utils.clip_grad_norm_(raw.parameters(), 1.0)
    opt.step()
    step += 1
    recent.append(loss.item()); recent = recent[-50:]
    tok_s = (step - s0) * BATCH * BLOCK / (time.time() - t0)
    if step % 50 == 0:
        left = (MAX_STEPS - step) * BATCH * BLOCK / tok_s / 60
        print(f'adım {step:6d} | loss {np.mean(recent):.4f} | lr {lr:.2e} | norm {norm:.2f} | {tok_s:,.0f} tok/sn | tur {epoch} | kalan ~{left:.0f} dk')
    if step % EVAL_EVERY == 0 or step == MAX_STEPS:
        evaluate(step, float(np.mean(recent)), tok_s)
    if step % CKPT_EVERY == 0 or step == MAX_STEPS:
        save_ckpt()
    if globals().get('SAVE_EVERY') and step % SAVE_EVERY == 0:        # 04c: her N adımda ayrı, küçük (bf16) checkpoint
        torch.save(dict(model={k: v.to(torch.bfloat16) for k, v in raw.state_dict().items()}, step=step, cfg=cfg),
                   f'{ROOT}/ckpt/{RUN}_step{step:05d}.pt')
print('bitti. checkpoint:', CKPT)

In [ ]:
# İstediğin cümleyi söylet
a = speak('Bu modeli sıfırdan ben eğittim.', seed=1)
save_wav(f'{ROOT}/samples/{RUN}/{RUN}_deneme.wav', a)
try:
    from IPython.display import Audio, display
    display(Audio(a, rate=SR))
except Exception: pass